<a href="https://colab.research.google.com/github/Muskan624-ai/Bias-detector/blob/main/notebooks/Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/BiasProject/data.csv')
print(df.head())
df = df.dropna()

                                    text  label
0   the media always lies about politics      1
1  the government announced a new policy      0
2   this law unfairly targets minorities      1
3         the meeting was held yesterday      0
4            this news channel is biased      1


In [ ]:
import re
from sklearn.model_selection import train_test_split

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

df['text'] = df['text'].apply(clean_text)

# Splitting the Data

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

print(f"Data Prepared!")
print(f"Total rows: {len(df)}")
print(f"Training on: {len(train_texts)} sentences")
print(f"Testing on: {len(test_texts)} sentences")

Data Prepared!
Total rows: 3600
Training on: 2880 sentences
Testing on: 720 sentences


In [ ]:
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# # Tokenizing the data
# # We add padding (to make all sentences the same length)
# # and truncation (to cut off very long sentences)
# train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
# test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

# print("Tokenization Complete!")
# print(f"Example of first sentence as IDs: {train_encodings['input_ids'][0][:10]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization Complete!
Example of first sentence as IDs: [101, 4518, 3006, 2701, 3020, 2651, 102, 0]...


In [ ]:
# import torch

# class BiasDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels.tolist()

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item['labels'] = torch.tensor(self.labels[idx])
#         return item

#     def __len__(self):
#         return len(self.labels)

# # Convert our data into the Torch format
# train_dataset = BiasDataset(train_encodings, train_labels)
# test_dataset = BiasDataset(test_encodings, test_labels)

# print("Datasets ready for the model!")

Datasets ready for the model!


In [ ]:
# from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# #2 labels: 0 (Neutral) and 1 (Biased)
# model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# # Move model to GPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# from transformers import TrainingArguments

# training_args = TrainingArguments(
#     output_dir='./results',          # Where to save the model
#     num_train_epochs=3,              # How many times to look at the data
#     per_device_train_batch_size=16,  # How many sentences to look at at once
#     per_device_eval_batch_size=16,
#     warmup_steps=500,                # Gradually increase learning rate
#     weight_decay=0.01,               # Prevents the model from "cheating" (overfitting)
#     eval_strategy="epoch",           # Changed from evaluation_strategy
#     logging_steps=10,
#     report_to="none"
# )

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.004463,0.003056
2,0.000550,0.000385
3,0.000168,0.000108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=540, training_loss=0.09351241814977214, metrics={'train_runtime': 39.7029, 'train_samples_per_second': 217.616, 'train_steps_per_second': 13.601, 'total_flos': 17883098818560.0, 'train_loss': 0.09351241814977214, 'epoch': 3.0})

In [ ]:
# # Create the Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset
# )
# trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000011,0.000006
2,0.000001,0.000001
3,0.000001,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=540, training_loss=2.3551144520044747e-05, metrics={'train_runtime': 30.9723, 'train_samples_per_second': 278.959, 'train_steps_per_second': 17.435, 'total_flos': 17883098818560.0, 'train_loss': 2.3551144520044747e-05, 'epoch': 3.0})

In [ ]:
# model.save_pretrained("./bias_model_final")
# tokenizer.save_pretrained("./bias_model_final")
# import shutil
# shutil.make_archive("bias_model_final", 'zip', "./bias_model_final")

# print("Model saved and zipped! Download 'bias_model_final.zip' from the folder icon on the left.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved and zipped! Download 'bias_model_final.zip' from the folder icon on the left.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_path = "/content/drive/MyDrive/BiasProject/bias_model_final"

# Load the saved model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Professional Handover Setup: Model loaded from permanent storage.")

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

✅ Professional Handover Setup: Model loaded from permanent storage.
